# Extract images/frames from video

In [ ]:
import cv2
import os

In [ ]:
vid = cv2.VideoCapture("E:\my computer\folder/vid.mp4")
currentframe = 0

In [ ]:
while (True):
    success , frame = vid.read()
    cv2.imshow("output" ,frame)
    cv2.imwrite("./data/frame" + str(currentframe) + '.jpg' , frame) # saves the frames in folder 'data' with name frame0.jpg frame1.jpg
    currentframe+ = 1
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

vid.release()
cv2.destroyALLWindows()

# Extract images/frames from WEBCAM

In [ ]:
import cv2
import os

In [ ]:
vid = cv2.VideoCapture(0)
currentframe = 0

In [ ]:
while (True):
    success , frame = vid.read()
    cv2.imshow("output" ,frame)
    cv2.imwrite("./data/frame" + str(currentframe) + '.jpg' , frame) # saves the frames in folder 'data' with name frame0.jpg frame1.jpg
    currentframe+ = 1
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

vid.release()
cv2.destroyALLWindows()

# Inference MLLM(MultimodalLLM) - LLaVA Phi3


# Install packages

In [ ]:
!git clone https://github.com/mbzuai-oryx/LLaVA-pp.git

In [ ]:
%cd LLaVA
!pip install --upgrade pip  > /dev/null
!pip install -e .  > /dev/null
!pip install git+https://github.com/huggingface/transformers@a98c41798cf6ed99e1ff17e3792d6e06a2ff2ff3  > /dev/null

!export PYTHONPATH="./:$PYTHONPATH"

# Import required packages

In [ ]:
import torch
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN, IMAGE_PLACEHOLDER
from llava.conversation import conv_templates
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import process_images, tokenizer_image_token, get_model_name_from_path
import requests
from PIL import Image
from io import BytesIO
import re
from llava.utils import disable_torch_init

# Define helper functions

In [ ]:
def image_parser(args):
    out = args.image_file.split(args.sep)
    return out


def load_image(image_file):
    if image_file.startswith("http") or image_file.startswith("https"):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_file).convert("RGB")
    return image


def load_images(image_files):
    out = []
    for image_file in image_files:
        image = load_image(image_file)
        out.append(image)
    return out

# Load LLaVA-Phi3

In [ ]:
disable_torch_init()

model_path = "/kaggle/input/llava-phi-3-mini-4k-instruct/transformers/1.0.0/1/llava-phi-3-mini-4k-instruct"
model_name = get_model_name_from_path(model_path)
tokenizer, model, image_processor, context_len = load_pretrained_model(model_path, None, model_name)

# Create the prompt in Phi-3 Format

In [ ]:
DEFAULT_IM_START_TOKEN

\<im_start>
DEFAULT_IMAGE_TOKEN

\<image>
DEFAULT_IM_END_TOKEN

\<im_end>

In [ ]:
qs = "Please describe the plant traits in detail. Please think step by step."
 
conv_mode = "phi3_instruct"

image_token_se = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN
if IMAGE_PLACEHOLDER in qs:
    if model.config.mm_use_im_start_end:
        qs = re.sub(IMAGE_PLACEHOLDER, image_token_se, qs)
    else:
        qs = re.sub(IMAGE_PLACEHOLDER, DEFAULT_IMAGE_TOKEN, qs)
else:
    if model.config.mm_use_im_start_end:
        qs = image_token_se + "\n" + qs
    else:
        qs = DEFAULT_IMAGE_TOKEN + "\n" + qs

conv = conv_templates[conv_mode].copy()
conv.system = """<|system|>\nYou are a helpful visual AI assistant."""
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

print(prompt)

In [ ]:
OUTPUT :

<|system|>
You are a helpful visual AI assistant.<|end|>
<|user|>
<image>
Please describe the plant traits in detail. Please think step by step.<|end|>
<|assistant|>

# Inference for plant iamge - Describing plants

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

saple_img_path = '/kaggle/input/planttraits2024/train_images/100438030.jpeg'

img = mpimg.imread(saple_img_path)
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
image_name = saple_img_path
image_files = [image_name]
images = load_images(image_files)
image_sizes = [x.size for x in images]
images_tensor = process_images(
    images,
    image_processor,
    model.config
).to(model.device, dtype=torch.float16)

input_ids = (
    tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
    .unsqueeze(0)
    .cuda()
)

In [ ]:
temperature = 0.2
top_p = 0.7
num_beams = 1
max_new_tokens = 512

with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        images=images_tensor,
        image_sizes=image_sizes,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        use_cache=True,
    )

outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
outputs = outputs.replace("<|end|>", "").strip()
outputs